Exploratory Data Analysis

In [250]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("airbnb.csv")


df.info()
df.describe()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27106 entries, 0 to 27105
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              27106 non-null  int64  
 1   name                            26557 non-null  object 
 2   host_id                         27106 non-null  int64  
 3   host_name                       26552 non-null  object 
 4   neighbourhood_group             26566 non-null  object 
 5   neighbourhood                   26571 non-null  object 
 6   latitude                        27106 non-null  float64
 7   longitude                       27106 non-null  float64
 8   room_type                       26570 non-null  object 
 9   price                           26595 non-null  float64
 10  minimum_nights                  26586 non-null  float64
 11  number_of_reviews               26583 non-null  float64
 12  last_review                     

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,27106.000000,2.710600e+04,27106.000000,27106.000000,26595.000000,26586.000000,26583.000000,21340.000000,26586.000000,26583.000000
mean,13692.979156,6.817564e+07,40.728885,-73.952079,164.604200,7.079687,21.255378,1.379605,9.594176,114.490026
std,7905.881696,7.911806e+07,0.054639,0.046297,6772.488296,614.394492,1369.984414,58.940016,1022.066006,4069.430794
min,1.000000,2.438000e+03,40.506410,-74.242850,-44039.906822,-3995.969493,-8836.739029,-341.215943,-6616.029612,-26299.960427
25%,6851.250000,7.958923e+06,40.690022,-73.983117,66.000000,1.000000,1.000000,0.180000,1.000000,0.000000
50%,13688.500000,3.104429e+07,40.723105,-73.955720,107.000000,3.000000,5.000000,0.730000,1.000000,46.000000
75%,20541.750000,1.074344e+08,40.763098,-73.935830,180.000000,5.000000,25.000000,2.170000,2.000000,242.500000
max,27379.000000,2.743213e+08,40.913060,-73.717950,44388.408761,4008.347956,8875.024307,343.979641,6637.886853,26518.501580


Checking for null columns.

In [251]:
print(df.describe(include='all'))

                  id            name       host_id host_name  \
count   27106.000000           26557  2.710600e+04     26552   
unique           NaN           26206           NaN      7846   
top              NaN  Hillside Hotel           NaN   Michael   
freq             NaN              15           NaN       243   
mean    13692.979156             NaN  6.817564e+07       NaN   
std      7905.881696             NaN  7.911806e+07       NaN   
min         1.000000             NaN  2.438000e+03       NaN   
25%      6851.250000             NaN  7.958923e+06       NaN   
50%     13688.500000             NaN  3.104429e+07       NaN   
75%     20541.750000             NaN  1.074344e+08       NaN   
max     27379.000000             NaN  2.743213e+08       NaN   

       neighbourhood_group neighbourhood      latitude     longitude  \
count                26566         26571  27106.000000  27106.000000   
unique                  72           283           NaN           NaN   
top            

In [252]:
print(df.isnull().sum())

id                                   0
name                               549
host_id                              0
host_name                          554
neighbourhood_group                540
neighbourhood                      535
latitude                             0
longitude                            0
room_type                          536
price                              511
minimum_nights                     520
number_of_reviews                  523
last_review                       5995
reviews_per_month                 5766
calculated_host_listings_count     520
availability_365                   523
dtype: int64


Checking for negative or zero values in numeric columns.

In [253]:
negative_prices = (df['price'] <= 0).sum()
negative_nights = (df['minimum_nights'] < 0).sum()
negative_reviews = (df['number_of_reviews'] < 0).sum()
negative_rpm = (df['reviews_per_month'] < 0).sum()
negative_host_listings = (df['calculated_host_listings_count'] < 0).sum()
negative_availability = (df['availability_365'] < 0).sum()

print(f"Listings with price <= 0: {negative_prices}")
print(f"Listings with minimum_nights < 0: {negative_nights}")
print(f"Listings with number_of_reviews < 0: {negative_reviews}")
print(f"Listings with reviews_per_month < 0: {negative_rpm}")
print(f"Listings with calculated_host_listings_count < 0: {negative_host_listings}")
print(f"Listings with availability_365 < 0: {negative_availability}")

Listings with price <= 0: 547
Listings with minimum_nights < 0: 542
Listings with number_of_reviews < 0: 545
Listings with reviews_per_month < 0: 539
Listings with calculated_host_listings_count < 0: 542
Listings with availability_365 < 0: 541


Checking Categorial columns if it contains non related information.

In [254]:
print((df['neighbourhood'].value_counts(normalize=True, dropna=False) * 100).round(4))




neighbourhood
Williamsburg          7.6625
Bedford-Stuyvesant    7.3379
Harlem                5.1317
Bushwick              4.9251
Upper West Side       3.9585
                       ...  
🦄_EXOTIC_37           0.0037
🦄_EXOTIC_5            0.0037
🦄_EXOTIC_0            0.0037
🦄_EXOTIC_43           0.0037
🦄_EXOTIC_12           0.0037
Name: proportion, Length: 284, dtype: float64


In [255]:
print((df['neighbourhood_group'].value_counts(normalize=True, dropna=False) * 100).round(4))

neighbourhood_group
Manhattan      43.3151
Brooklyn       40.1756
Queens         11.3665
Bronx           2.1730
NaN             1.9922
                ...   
🦄_EXOTIC_16     0.0037
🦄_EXOTIC_33     0.0037
🦄_EXOTIC_26     0.0037
🦄_EXOTIC_61     0.0037
🦄_EXOTIC_1      0.0037
Name: proportion, Length: 73, dtype: float64


In [256]:
print((df['room_type'].value_counts(normalize=True, dropna=False)*100).round(4))

room_type
Entire home/apt    50.9186
Private room       44.5621
Shared room         2.2910
NaN                 1.9774
🦄_EXOTIC_17         0.0037
                    ...   
🦄_EXOTIC_5          0.0037
🦄_EXOTIC_32         0.0037
🦄_EXOTIC_33         0.0037
🦄_EXOTIC_45         0.0037
🦄_EXOTIC_16         0.0037
Name: proportion, Length: 72, dtype: float64


Data Preprocessing

Cleaning the dataset by removing rows with invalid numerical values.

In [257]:
invalid_price_mask = df['price'] <= 0

negative_columns = [
    'minimum_nights', 'number_of_reviews', 'reviews_per_month','calculated_host_listings_count', 'availability_365']

negative_value_mask = (df[negative_columns] < 0).any(axis=1)

df = df[~(invalid_price_mask | negative_value_mask)].copy()


In [258]:
df = df[df['availability_365'] <= 365].copy()

Removing non useful information from categorial columns.

In [259]:

filter = len(df) * (0.0001)

categorical = ['neighbourhood_group', 'room_type','neighbourhood']

for col in categorical:
    value_counts = df[col].value_counts()

    # Identify the categories that meet or exceed the 0.1% threshold
    categories_to_keep = value_counts[value_counts >= filter].index

    # Filter the DataFrame to keep those categories, explicitly preserving NaN values for now
    df = df[df[col].isin(categories_to_keep) | df[col].isnull()].copy()

In [260]:
df = df.dropna().copy()

Checking if the unwanted value has been dropped.

In [261]:
print((df['room_type'].value_counts(normalize=True, dropna=False)*100).round(4))

room_type
Entire home/apt    52.4829
Private room       45.4527
Shared room         2.0645
Name: proportion, dtype: float64


Dropping unneccessary columns.

In [262]:
columns_to_drop = ['id', 'host_id', 'name', 'host_name', 'last_review']

df = df.drop(columns_to_drop, axis=1).copy()

Removing Outliers using IQR.

In [263]:
numerical_cols_iqr = ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month','calculated_host_listings_count', 'availability_365']

outlier_mask = pd.Series(True, index=df.index)

for col in numerical_cols_iqr:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    col_mask = (df[col] >= lower_bound) & (df[col] <= upper_bound)
    outlier_mask = outlier_mask & col_mask

df_no_outliers = df[outlier_mask].copy()

Train Test Split

In [264]:
X = df_no_outliers.drop('price', axis=1)
y = df_no_outliers['price']
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3,random_state=67381600)



Scaling and Encoding using Scikit Learn Pipilines

In [265]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='passthrough'
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

Model Building & Evaluation

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

Training Tree Based Model(Decision Tree Regressor)

In [277]:
rfr_model = RandomForestRegressor(n_estimators = 200,  max_depth = 13, min_samples_split = 25,random_state = 67381600, n_jobs=-1)
rfr_model.fit(X_train, y_train)

y_pred_rfr = rfr_model.predict(X_test)

rfr_mse = mean_squared_error(y_test, y_pred_rfr)
rfr_r2 = r2_score(y_test, y_pred_rfr)

print("--- Random Forest Model Evaluation ---")
print(f"Mean Squared Error (MSE): {rfr_mse:.2f}")
print(f"R-squared (R2): {rfr_r2:.4f}")


feature_names_out = preprocessor.get_feature_names_out()
importances = rfr_model.feature_importances_
feature_importances_df = pd.DataFrame({
    'Feature': feature_names_out,
    'Importance': importances
})
top_10_features = feature_importances_df.sort_values(by='Importance', ascending=False).head(10)

print(top_10_features.to_string(index=False))

--- Random Forest Model Evaluation ---
Mean Squared Error (MSE): 2210.20
R-squared (R2): 0.5030
                            Feature  Importance
     cat__room_type_Entire home/apt    0.522903
                     num__longitude    0.159442
                      num__latitude    0.109713
              num__availability_365    0.066140
             num__reviews_per_month    0.035479
             num__number_of_reviews    0.027206
                num__minimum_nights    0.016167
    cat__neighbourhood_Williamsburg    0.008703
num__calculated_host_listings_count    0.005818
 cat__neighbourhood_group_Manhattan    0.004693


Training Regression Based Model (Lasso)

In [273]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, make_scorer

r2_scorer = make_scorer(r2_score)

param_grid = {
    'alpha': np.logspace(-4, 0, 20)
}

lasso = Lasso(random_state=67381600, max_iter=1000)

grid_search = GridSearchCV(
    estimator=lasso,
    param_grid=param_grid,
    scoring=r2_scorer,
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_lasso_model = grid_search.best_estimator_
best_params = grid_search.best_params_

print("--- Lasso Model Evaluation ---")

y_pred_best_lasso = best_lasso_model.predict(X_test)
best_lasso_r2 = r2_score(y_test, y_pred_best_lasso)
best_lasso_mse = mean_squared_error(y_test, y_pred_best_lasso)

feature_names_out = preprocessor.get_feature_names_out()
coefficients_df = pd.DataFrame({
    'Feature': feature_names_out,
    'Coefficient': best_lasso_model.coef_
})

significant_coefficients = coefficients_df[coefficients_df['Coefficient'] != 0]
significant_coefficients['Abs_Coefficient'] = significant_coefficients['Coefficient'].abs()
top_10_lasso = significant_coefficients.sort_values(by='Abs_Coefficient', ascending=False).head(10)
print(f"Mean Squared Error (MSE): {best_lasso_mse:.2f}")
print(f"R-squared (R2): {best_lasso_r2:.4f}")
print(top_10_lasso.drop(columns=['Abs_Coefficient']).to_string(index=False))

--- Lasso Model Evaluation ---
Mean Squared Error (MSE): 2261.61
R-squared (R2): 0.4914
                               Feature  Coefficient
        cat__room_type_Entire home/apt    75.163780
cat__neighbourhood_group_Staten Island   -47.451177
cat__neighbourhood_Springfield Gardens    45.198645
             cat__neighbourhood_Inwood   -44.839246
   cat__neighbourhood_Brooklyn Heights    40.614940
 cat__neighbourhood_Washington Heights   -40.228194
    cat__neighbourhood_group_Manhattan    37.985932
        cat__neighbourhood_South Slope    32.853559
  cat__neighbourhood_Downtown Brooklyn    29.614811
            cat__neighbourhood_Tribeca    28.835874


C:\Users\Art\AppData\Local\Temp\ipykernel_32252\2897918875.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  significant_coefficients['Abs_Coefficient'] = significant_coefficients['Coefficient'].abs()
